# DeceptiScope v2 — Deception Probe Training

This notebook trains and evaluates linear deception probes on hidden-state activations
extracted from open-weight models (LLaMA 3.1 8B, Mistral 7B).

## Key Research Questions
1. **Which layers encode deceptive intent?** — We expect middle layers (12–20 of 32) to show the strongest signal.
2. **How well does a linear probe generalise?** — Cross-benchmark AUROC.
3. **Does the probe transfer across deception types?** — Train on factual errors, test on sycophancy.

## Methodology
- Extract residual stream activations at every layer for honest vs. deceptive completions
- Train a logistic regression probe per layer
- Identify the 'deception direction' via PCA / contrastive mean difference
- Evaluate on held-out TruthfulQA + custom DeceptiScope benchmark

In [ ]:
import sys
sys.path.insert(0, '/app')  # Add backend to path

import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, classification_report
from sklearn.model_selection import cross_val_score

from whitebox.extractor import ActivationExtractor
from whitebox.probe import DeceptionProbe
from data.dataset_builder import DeceptionDatasetBuilder, DatasetConfig

# Plotting style
plt.style.use('dark_background')
sns.set_palette('husl')

print('DeceptiScope v2 — Probe Training Notebook')
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')

## 1. Load Model and Extract Activations

In [ ]:
# Configuration
MODEL_NAME = 'meta-llama/Llama-3.1-8B-Instruct'  # or 'mistralai/Mistral-7B-Instruct-v0.3'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
N_LAYERS = 32  # LLaMA 3.1 8B has 32 transformer layers

print(f'Loading {MODEL_NAME} on {DEVICE}...')

extractor = ActivationExtractor(
    model_name=MODEL_NAME,
    device=DEVICE,
    layers=list(range(N_LAYERS)),
    extract_residual=True,
    extract_attention=False,  # Focus on residual stream for probing
)

print(f'Model loaded. Hidden dim: {extractor.hidden_size}')

In [ ]:
# Load TruthfulQA dataset for probe training
# We use honest completions as label=0, deceptive as label=1

from datasets import load_dataset

truthfulqa = load_dataset('truthful_qa', 'generation', split='validation')
print(f'TruthfulQA: {len(truthfulqa)} examples')
print(truthfulqa[0])

In [ ]:
# Extract activations for honest and deceptive completions
# This takes ~20 min on A100 for 817 examples × 32 layers

honest_activations = []    # shape: (N, n_layers, hidden_dim)
deceptive_activations = [] # shape: (N, n_layers, hidden_dim)

SAMPLE_SIZE = 200  # Reduce for quick testing; use full 817 for paper

for i, example in enumerate(truthfulqa.select(range(SAMPLE_SIZE))):
    question = example['question']
    
    # Honest: best answer
    honest_text = f"Q: {question}\nA: {example['best_answer']}"
    h_acts = extractor.extract(honest_text)  # dict: layer -> tensor
    honest_activations.append([h_acts[l].cpu().numpy() for l in range(N_LAYERS)])
    
    # Deceptive: worst (most incorrect) answer
    if example['incorrect_answers']:
        deceptive_text = f"Q: {question}\nA: {example['incorrect_answers'][0]}"
        d_acts = extractor.extract(deceptive_text)
        deceptive_activations.append([d_acts[l].cpu().numpy() for l in range(N_LAYERS)])
    
    if (i + 1) % 50 == 0:
        print(f'Extracted {i+1}/{SAMPLE_SIZE} examples')

print(f'Honest: {len(honest_activations)}, Deceptive: {len(deceptive_activations)}')

## 2. Train Per-Layer Probes

In [ ]:
# Train a logistic regression probe at each layer
# Key finding: probe accuracy peaks at middle layers

layer_aucs = []
layer_probes = []

n = min(len(honest_activations), len(deceptive_activations))

for layer_idx in range(N_LAYERS):
    # Build feature matrix: last-token activation at this layer
    X_honest    = np.array([honest_activations[i][layer_idx][-1] for i in range(n)])
    X_deceptive = np.array([deceptive_activations[i][layer_idx][-1] for i in range(n)])
    
    X = np.vstack([X_honest, X_deceptive])
    y = np.array([0] * n + [1] * n)
    
    # Train probe with cross-validation
    probe = LogisticRegression(C=0.1, max_iter=1000, solver='lbfgs')
    auc_scores = cross_val_score(probe, X, y, cv=5, scoring='roc_auc')
    
    layer_aucs.append(auc_scores.mean())
    probe.fit(X, y)
    layer_probes.append(probe)
    
    if layer_idx % 8 == 0:
        print(f'Layer {layer_idx:2d}: AUC = {auc_scores.mean():.3f} ± {auc_scores.std():.3f}')

peak_layer = np.argmax(layer_aucs)
print(f'\nPeak layer: {peak_layer} (AUC = {layer_aucs[peak_layer]:.3f})')

In [ ]:
# Visualise per-layer probe AUC
# This is Figure 1 in the grant application

fig, ax = plt.subplots(figsize=(12, 4))

colors = ['#f85149' if auc > 0.75 else '#d29922' if auc > 0.6 else '#3fb950' 
          for auc in layer_aucs]

bars = ax.bar(range(N_LAYERS), layer_aucs, color=colors, alpha=0.85, edgecolor='none')
ax.axhline(0.5, color='#8b949e', linestyle='--', alpha=0.5, label='Random baseline')
ax.axhline(0.75, color='#f85149', linestyle=':', alpha=0.5, label='High-signal threshold')
ax.axvline(peak_layer, color='#bc8cff', linestyle='-', alpha=0.7, linewidth=2,
           label=f'Peak layer {peak_layer}')

ax.set_xlabel('Transformer Layer', fontsize=12)
ax.set_ylabel('Probe AUC-ROC', fontsize=12)
ax.set_title('Deception Probe Performance by Layer\n'
             '(LLaMA 3.1 8B, TruthfulQA)', fontsize=13)
ax.set_ylim(0.4, 1.0)
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.2)

plt.tight_layout()
plt.savefig('/notebooks/figures/layer_probe_auc.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Interpretability insight: Layer {peak_layer} is where the model "knows" the truth.')
print('This is the optimal layer for RepE steering vector extraction.')

## 3. Extract Deception Direction (RepE)

In [ ]:
# Extract the deception direction at the peak layer
# Method: contrastive mean difference (honest_mean - deceptive_mean)
# This is the vector we add to activations to steer toward honesty

layer_idx = peak_layer

honest_vecs    = np.array([honest_activations[i][layer_idx][-1] for i in range(n)])
deceptive_vecs = np.array([deceptive_activations[i][layer_idx][-1] for i in range(n)])

# Deception direction: deceptive - honest (points toward deception)
deception_direction = deceptive_vecs.mean(0) - honest_vecs.mean(0)
deception_direction /= np.linalg.norm(deception_direction)  # Normalise

# Honest direction: opposite
honest_direction = -deception_direction

print(f'Deception direction shape: {deception_direction.shape}')
print(f'Norm: {np.linalg.norm(deception_direction):.4f} (should be 1.0)')

# Project activations onto deception direction
honest_proj    = honest_vecs @ deception_direction
deceptive_proj = deceptive_vecs @ deception_direction

print(f'\nProjection stats:')
print(f'  Honest:    mean={honest_proj.mean():.3f}, std={honest_proj.std():.3f}')
print(f'  Deceptive: mean={deceptive_proj.mean():.3f}, std={deceptive_proj.std():.3f}')
print(f'  Separation: {deceptive_proj.mean() - honest_proj.mean():.3f} std units')

In [ ]:
# Visualise the deception direction separation
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Distribution plot
axes[0].hist(honest_proj, bins=30, alpha=0.7, color='#3fb950', label='Honest', density=True)
axes[0].hist(deceptive_proj, bins=30, alpha=0.7, color='#f85149', label='Deceptive', density=True)
axes[0].set_xlabel('Projection onto deception direction')
axes[0].set_ylabel('Density')
axes[0].set_title(f'Activation Separation at Layer {peak_layer}')
axes[0].legend()
axes[0].grid(alpha=0.2)

# PCA scatter
from sklearn.decomposition import PCA
all_vecs = np.vstack([honest_vecs, deceptive_vecs])
pca = PCA(n_components=2)
pca_coords = pca.fit_transform(all_vecs)

axes[1].scatter(pca_coords[:n, 0], pca_coords[:n, 1], 
                c='#3fb950', alpha=0.5, s=20, label='Honest')
axes[1].scatter(pca_coords[n:, 0], pca_coords[n:, 1], 
                c='#f85149', alpha=0.5, s=20, label='Deceptive')
axes[1].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
axes[1].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
axes[1].set_title(f'PCA of Layer {peak_layer} Activations')
axes[1].legend()
axes[1].grid(alpha=0.2)

plt.tight_layout()
plt.savefig('/notebooks/figures/deception_direction.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Save Probe and Direction for Production Use

In [ ]:
import pickle
import os

os.makedirs('/models/probes', exist_ok=True)

# Save best probe
best_probe = layer_probes[peak_layer]
with open(f'/models/probes/deception_probe_layer{peak_layer}.pkl', 'wb') as f:
    pickle.dump(best_probe, f)

# Save deception direction
np.save(f'/models/probes/deception_direction_layer{peak_layer}.npy', deception_direction)

# Save all layer AUCs for the dashboard
np.save('/models/probes/layer_aucs.npy', np.array(layer_aucs))

print(f'Saved probe and direction for layer {peak_layer}')
print(f'Peak AUC: {layer_aucs[peak_layer]:.3f}')
print(f'All layer AUCs saved to /models/probes/layer_aucs.npy')